# Can we predict `passed_quiz`?
Each row is one tutoring session; the target is whether the student passed the closing quiz.

**Plan**
1. Check the data (repeat students, odd values).
2. Evaluate honestly: the same student appears in many sessions, so a random split would leak student identity. We use **grouped** cross-validation (whole students held out), i.e. predicting for students the model has never seen.
3. Compare against a trivial baseline, a logistic regression and gradient boosting.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv'); df.shape

## 1. Data checks

In [ ]:
print(df.isna().sum().sum(), 'missing;', df.duplicated().sum(), 'duplicate rows')
print('sessions per student:'); print(df.groupby('student_key').size().describe().round(1))
print('base rate of passing:', df.passed_quiz.mean().round(3))
print('prior_score > 100:', (df.prior_score>100).sum(), '| range', df.prior_score.min(), df.prior_score.max())
print('students with >1 distinct prior_score:', (df.groupby('student_key').prior_score.nunique()>1).mean().round(3))
df.describe().round(2)

Observations:
- 1,800 students, ~7 sessions each; `enrol_year` and `school_code` are constant per student.
- `prior_score` is described as a placement test score, yet it **varies between a student's sessions** (it is stored on the session record), so it behaves like a noisy per-session value rather than a fixed trait. A few values exceed 100; kept as is.
- Pass rates differ between students (below) more than chance would give, so student identity matters and ignoring it in the split would inflate results.

In [ ]:
sm = df.groupby('student_key').passed_quiz.agg(['mean','size'])
expected_sd = np.sqrt((sm['mean'].mean()*(1-sm['mean'].mean())/sm['size']).mean())
print('SD of student pass rates: observed %.3f vs %.3f expected if sessions were independent of student' % (sm['mean'].std(), expected_sd))
fig,ax=plt.subplots(1,3,figsize=(14,3.5))
df.groupby(pd.cut(df.session_minutes,[0,10,20,30,45,60,90,210]), observed=True).passed_quiz.mean().plot.bar(ax=ax[0],title='pass rate by session_minutes')
df.groupby('hints_used').passed_quiz.mean().plot.bar(ax=ax[1],title='pass rate by hints_used')
df.groupby(pd.cut(df.prior_score,[0,40,50,60,70,80,110]), observed=True).passed_quiz.mean().plot.bar(ax=ax[2],title='pass rate by prior_score')
for a in ax: a.set_ylim(0,0.7); a.set_xlabel('')
plt.tight_layout()

## 2. Models (grouped 5-fold CV by student)

In [ ]:
feats = ['enrol_year','school_code','prior_score','session_minutes','hints_used']
X, y, g = df[feats], df.passed_quiz, df.student_key
cv = GroupKFold(n_splits=5)
models = {
 'baseline (base rate)': DummyClassifier(strategy='prior'),
 'logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
 'gradient boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
rows, oof = [], {}
for n,m in models.items():
    p = cross_val_predict(m, X, y, groups=g, cv=cv, method='predict_proba')[:,1]; oof[n]=p
    rows.append(dict(model=n, AUC=roc_auc_score(y,p), logloss=log_loss(y,p), brier=brier_score_loss(y,p), accuracy=((p>.5)==y).mean()))
pd.DataFrame(rows).set_index('model').round(3)

`school_code` is a numeric code, not an ordered quantity; treating it as a number is a simplification, but it carries almost no signal (see below).

In [ ]:
aucs=[roc_auc_score(y.iloc[te], oof['logistic regression'][te]) for _,te in cv.split(X,y,g)]
print('logistic AUC per fold:', np.round(aucs,3), '| mean %.3f sd %.3f'%(np.mean(aucs),np.std(aucs)))
lr = models['logistic regression'].fit(X,y)
print(pd.Series(lr[-1].coef_[0], feats).round(3).rename('std. coefficient'))
gb = models['gradient boosting'].fit(X,y)
pi = permutation_importance(gb, X, y, scoring='roc_auc', n_repeats=5, random_state=0)
print(pd.Series(pi.importances_mean, feats).round(4).sort_values(ascending=False).rename('GB permutation importance (AUC drop, in-sample)'))

## 3. Calibration

In [ ]:
p = oof['logistic regression']
q = pd.qcut(p, 10, duplicates='drop')
t = pd.DataFrame({'pred':p,'obs':y}).groupby(q, observed=True).mean()
plt.figure(figsize=(4.5,4)); plt.plot(t.pred,t.obs,'o-'); plt.plot([.2,.7],[.2,.7],'k--',lw=1)
plt.xlabel('mean predicted'); plt.ylabel('observed pass rate'); plt.title('Calibration by decile'); plt.show()
print('observed pass rate, lowest vs highest predicted decile: %.2f vs %.2f' % (t.obs.iloc[0], t.obs.iloc[-1]))

## Conclusions
Numbers from a run of this notebook (5-fold CV, whole students held out):

- **Weak but real signal.** Logistic regression reaches AUC ≈ 0.60 (±0.01 across folds), accuracy ≈ 0.60 vs 0.58 for always predicting "fail". Gradient boosting is no better (AUC ≈ 0.58), so there is no sign of useful non-linear structure in these 5 features.
- **What drives it:** longer sessions → more likely to pass; more hints used → less likely. `prior_score`, `enrol_year` and `school_code` add essentially nothing (the boosting model's in-sample importance for `school_code` is overfitting; it does not show up in the linear model or in CV).
- **Calibration is fine** (predicted ≈ observed), and the extremes separate (≈ 28% vs ≈ 57% pass rate between lowest and highest predicted decile). That is useful for ranking sessions, not for confidently predicting any individual outcome.
- **Unexplained student effect.** Pass rates vary across students far more than the features explain (SD 0.35 vs 0.20 expected by chance). A student's history would probably help for returning students, but it is not usable for new students, and the data has no session dates, so I did not build history features (they would use "future" sessions).

**Caveats:** `session_minutes` and `hints_used` are measured during the session, so they are available only at or near its end, not before it starts. `prior_score` varying within a student is odd for a placement score and is worth checking with whoever owns the data.